# 07 — Performance Efficiency & Cost Optimization

The two WAF pillars that most directly show up on the cloud bill.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Performance Efficiency** | **Predictive Optimization** removes OPTIMIZE + VACUUM from your roadmap entirely — Databricks decides when to run them based on real query patterns, and you stop paying for maintenance jobs that were doing the same work. |
| **Cost Optimization** | The `cost_center` tag planted in notebook 00 flows into `system.billing.usage.custom_tags` — a tag-based chargeback report is a single SQL query away. **This is the slide for the CFO.** |
| **Cost Optimization (cleanup)** | The `removeAfter` tag + a 5-line scanner query = automated sandbox garbage collection for the whole workspace. Plug it into Lakeflow and you never again have an "abandoned demo catalog" problem. |
| **Operational Excellence** | The warehouse-sizing query reads `system.query.history` to surface queries whose p99 latency is bad — a continuous loop that tells you when to resize a warehouse up *or* down. |


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
COST_CENTER   = os.getenv("WAF_COST_CENTER", "field_engineering")

print(f"Cost center: {COST_CENTER}")


Cost center: field_engineering


## 1. Predictive optimization — turn on auto-optimize and auto-vacuum

Predictive Optimization runs OPTIMIZE + VACUUM on your behalf for tables in the
schema. Turning it on at the schema level is the WAF-recommended default —
you stop paying for maintenance jobs that were doing the same work.

In [2]:
# Read current state (requires the relevant privilege in your workspace).
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    try:
        row = spark.sql(f"DESCRIBE SCHEMA EXTENDED {UC_CATALOG}.{schema}").collect()
        for r in row:
            if r["database_description_item"] == "Predictive Optimization":
                print(f"  {schema}: {r['database_description_value']}")
                break
    except Exception as e:
        print(f"  {schema}: could not read ({str(e)[:80]})")


  mlb_gumbo_waf_bronze: ENABLE


  mlb_gumbo_waf_silver: ENABLE


  mlb_gumbo_waf_gold: ENABLE


In [3]:
# Enable predictive optimization for the three demo schemas. This is
# idempotent — safe to re-run. Requires the account setting that enables PO.
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    try:
        spark.sql(f"ALTER SCHEMA {UC_CATALOG}.{schema} ENABLE PREDICTIVE OPTIMIZATION")
        print(f"  PO enabled: {schema}")
    except Exception as e:
        print(f"  {schema}: {str(e)[:160]}")


  PO enabled: mlb_gumbo_waf_bronze


  PO enabled: mlb_gumbo_waf_silver


  PO enabled: mlb_gumbo_waf_gold


## 2. Inspect liquid clustering on the hot table

`fact_pitches` is our high-cardinality, high-read-rate table. We clustered on
`(season, official_date, pitcher_sk)` — the exact predicate shape that the
dashboard and pitcher-leaderboard queries emit. Below is how you verify that.

In [4]:
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"

print("fact_pitches clustering keys (from DESCRIBE DETAIL):")
spark.sql(f"DESCRIBE DETAIL {G}.fact_pitches").select(
    "name", "clusteringColumns", "numFiles", "sizeInBytes"
).show(truncate=False)

print("\nTop 5 largest files and their approximate season/date range:")
spark.sql(f"DESCRIBE HISTORY {G}.fact_pitches").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(5, truncate=False)


fact_pitches clustering keys (from DESCRIBE DETAIL):


+-----------------------------------------------+-----------------------------------+--------+-----------+
|name                                           |clusteringColumns                  |numFiles|sizeInBytes|
+-----------------------------------------------+-----------------------------------+--------+-----------+
|alexander_booth.mlb_gumbo_waf_gold.fact_pitches|[season, official_date, pitcher_sk]|1       |41722253   |
+-----------------------------------------------+-----------------------------------+--------+-----------+


Top 5 largest files and their approximate season/date range:


+-------+-------------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation              |operationMetrics                                                                                                                                |
+-------+-------------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |2026-04-21 20:55:38|SET TBLPROPERTIES      |{}                                                                                                                                              |
|2      |2026-04-21 20:55:17|CHANGE COLUMN          |{}                                                                                                                                              |
|1   

## 3. Tag-based chargeback from `system.billing.usage`

This is the moment the CFO cares about. Because the catalog + schemas + tables
all carry the `cost_center` tag we set in notebook 00, every DBU charge
associated with this demo automatically shows up under that cost center in
`system.billing.usage.custom_tags`.

In [5]:
print("DBUs by workload tag for this demo (last 30 days):")
try:
    spark.sql(f"""
        SELECT
            custom_tags['cost_center']  AS cost_center,
            custom_tags['env']          AS env,
            usage_metadata.job_id       AS job_id,
            billing_origin_product      AS product,
            SUM(usage_quantity)         AS dbus
        FROM system.billing.usage
        WHERE usage_date >= current_date() - INTERVAL 30 DAYS
          AND custom_tags['cost_center'] = '{COST_CENTER}'
        GROUP BY 1, 2, 3, 4
        ORDER BY dbus DESC
        LIMIT 20
    """).show(truncate=False)
except Exception as e:
    print(f"  Could not query system.billing.usage: {str(e)[:160]}")
    print("  Make sure system tables are enabled for this workspace.")


DBUs by workload tag for this demo (last 30 days):


+-----------------+----+---------------+-------+--------------------+
|cost_center      |env |job_id         |product|dbus                |
+-----------------+----+---------------+-------+--------------------+
|field_engineering|NULL|493033955038137|JOBS   |0.019249241666666667|
+-----------------+----+---------------+-------+--------------------+



## 4. Cost hygiene: `removeAfter` stale-asset detector

We put `removeAfter` tags on the catalog so old demo sandboxes can be garbage-collected
automatically. This query is a generic scanner — plug it into a Lakeflow job
and you have automated cleanup for the whole workspace.

In [6]:
print("Any asset in this catalog whose 'removeAfter' date is in the past:")
try:
    spark.sql(f"""
        SELECT schema_name, table_name, tag_value AS remove_after
        FROM {UC_CATALOG}.information_schema.table_tags
        WHERE tag_name = 'removeAfter'
          AND TO_DATE(tag_value, 'yyyyMMdd') < current_date()
        ORDER BY tag_value
    """).show(50, truncate=False)
except Exception as e:
    print(f"  Could not read information_schema.table_tags: {str(e)[:160]}")
    print("  (Query the per-catalog information_schema — system.information_schema")
    print("   doesn't have a table_tags view on all workspaces.)")


Any asset in this catalog whose 'removeAfter' date is in the past:


+-----------+----------+------------+
|schema_name|table_name|remove_after|
+-----------+----------+------------+
+-----------+----------+------------+



## 5. Serverless SQL warehouse sizing recommendation

This final query looks at queries that ran against your SQL warehouses and flags
ones that would have benefitted from a smaller (or larger) t-shirt size — the
simplest continuous cost-optimization loop.

In [7]:
print("Recent query performance — 95th pct durations by warehouse:")
try:
    spark.sql("""
        SELECT
            warehouse_id,
            COUNT(*) AS queries,
            ROUND(PERCENTILE(total_duration_ms, 0.5)  / 1000.0, 2) AS p50_s,
            ROUND(PERCENTILE(total_duration_ms, 0.95) / 1000.0, 2) AS p95_s,
            ROUND(PERCENTILE(total_duration_ms, 0.99) / 1000.0, 2) AS p99_s
        FROM system.query.history
        WHERE start_time >= current_timestamp() - INTERVAL 7 DAYS
          AND warehouse_id IS NOT NULL
        GROUP BY warehouse_id
        ORDER BY queries DESC
        LIMIT 10
    """).show(truncate=False)
except Exception as e:
    print(f"  Could not read query history: {str(e)[:160]}")


Recent query performance — 95th pct durations by warehouse:


{"ts": "2026-04-21 15:57:05.186", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `warehouse_id` cannot be resolved. Did you mean one of the following? [`account_id`, `workspace_id`, `session_id`, `compute`, `executed_as`]. SQLSTATE: 42703; line 10 pos 14;\n'GlobalLimit 10\n+- 'LocalLimit 10\n   +- 'Sort ['queries DESC NULLS LAST], true\n      +- 'Aggregate ['warehouse_id], ['warehouse_id, count(1) AS queries#339808L, 'ROUND(('PERCENTILE('total_duration_ms, 0.5) / 1000.0), 2) AS p50_s#339809, 'ROUND(('PERCENTILE('total_duration_ms, 0.95) / 1000.0), 2) AS p95_s#339810, 'ROUND(('PERCENTILE('total_duration_ms, 0.99) / 1000.0), 2) AS p99_s#339811]\n         +- 'Filter ((start_time#339833 >= cast(current_timestamp() - INTERVAL '7' DAY as timestamp)) AND isnotnull('warehouse_id))\n            +- SubqueryAlias system.query.history\n               +- Relation system.query.history[account_id#339

  Could not read query history: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `warehouse_id` cannot be resolved. Did you mean one of the following? [`
